Build the training set from historical matches.

Each row uses only information known before kickoff (pre-match Elo, venue, neutral, tournament weight) plus the final score as the target. Post-match Elo is never written to this file.

A match is competitive when the tournament is not a Friendly. Friendlies still update Elo, then are dropped from training.

In [5]:
import pandas as pd
from collections import defaultdict

historical_matches = pd.read_csv("../data/processed/historical_matches.csv", index_col=0)

non_fifa_tournaments = [
    "Viva World Cup", "FIFI Wild Cup", "ELF Cup", "World Unity Cup",
    "Atlantic Heritage Cup", "Hungary Heritage Cup", "Benedikt Fontana Cup",
    "Tynwald Hill Tournament", "Island Games", "Inter Games", "Muratti Vase",
    "Coupe de l'Outre-Mer",
]
historical_matches = historical_matches[~historical_matches["tournament"].isin(non_fifa_tournaments)]
historical_matches = historical_matches[~historical_matches["tournament"].str.contains("CONIFA", case=False)]
historical_matches = historical_matches.dropna(subset=["home_score", "away_score"])

print(historical_matches.shape)
historical_matches.head()

(48272, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


Tournament tiers — same buckets used for Elo K-factor. Re-defined here because notebooks don't share state.

In [6]:
TIER_1_WORLD_CUP = {"FIFA World Cup"}

TIER_2_CONTINENTAL = {
    "UEFA Euro", "Copa América", "African Cup of Nations", "AFC Asian Cup",
    "Gold Cup", "CONCACAF Championship", "Oceania Nations Cup",
    "Confederations Cup",
}

TIER_3_QUALIFIERS_NATIONS = {
    "FIFA World Cup qualification", "UEFA Euro qualification",
    "African Cup of Nations qualification", "AFC Asian Cup qualification",
    "Gold Cup qualification", "CONCACAF Championship qualification",
    "Copa América qualification", "Oceania Nations Cup qualification",
    "UEFA Nations League", "CONCACAF Nations League",
    "CONCACAF Nations League qualification",
}

TIER_4_REGIONAL = {
    # Africa
    "CECAFA Cup", "COSAFA Cup", "COSAFA Cup qualification", "WAFF Championship",
    "Amílcar Cabral Cup", "All-African Games", "UDEAC Cup", "UNIFFAC Cup",
    "West African Cup", "Nile Basin Tournament", "African Friendship Games",
    # Asia / Oceania
    "Gulf Cup", "Arab Cup", "Arab Cup qualification", "SAFF Cup",
    "AFF Championship", "AFF Championship qualification", "EAFF Championship",
    "EAFF Championship qualification", "ASEAN Championship",
    "ASEAN Championship qualification", "AFC Challenge Cup",
    "AFC Challenge Cup qualification", "Asian Games", "CAFA Nations Cup",
    "Southeast Asian Games", "South Asian Games", "Dynasty Cup",
    "Pacific Games", "South Pacific Games", "Melanesia Cup",
    "Indian Ocean Island Games", "Afro-Asian Games",
    # Europe
    "British Home Championship", "Nordic Championship", "Baltic Cup",
    "Balkan Cup", "Central European International Cup",
    # Americas
    "CFU Caribbean Cup", "CFU Caribbean Cup qualification", "UNCAF Cup",
    "Central American and Caribbean Games", "Pan American Championship",
    "CCCF Championship", "Bolivarian Games", "NAFC Championship",
    # Multi-sport
    "Olympic Games",
}

def tournament_weight(t):
    if t in TIER_1_WORLD_CUP:
        return 5
    if t in TIER_2_CONTINENTAL:
        return 4
    if t in TIER_3_QUALIFIERS_NATIONS:
        return 3
    if t in TIER_4_REGIONAL:
        return 2
    return 1  # friendlies + minor exhibitions

historical_matches["tournament_weight"] = historical_matches["tournament"].apply(tournament_weight)
print(historical_matches["tournament_weight"].value_counts().sort_index())

tournament_weight
1    21048
2     6692
3    16177
4     3391
5      964
Name: count, dtype: int64


Walk matches in date order. Snapshot each team's Elo before the result is applied, then update ratings for the next game only. K-factor is `10 * (tournament_weight + 1)` so the same tiers drive both the training feature and the Elo update.

In [7]:
HOME_ADVANTAGE = 100
START_ELO = 1500

def expected_score(home_elo, away_elo, neutral):
    dr = home_elo + (0 if neutral else HOME_ADVANTAGE) - away_elo
    return 1 / (1 + 10 ** (-dr / 400))

def actual_score(home_score, away_score):
    if home_score > away_score:
        return 1.0
    if home_score < away_score:
        return 0.0
    return 0.5

def goal_multiplier(home_score, away_score):
    gd = abs(home_score - away_score)
    if gd <= 1:
        return 1.0
    if gd == 2:
        return 1.5
    return (11 + gd) / 8

def elo_delta(home_elo, away_elo, home_score, away_score, k, neutral):
    expected = expected_score(home_elo, away_elo, neutral)
    actual = actual_score(home_score, away_score)
    return k * goal_multiplier(home_score, away_score) * (actual - expected)

matches = historical_matches.sort_values("date").copy()
matches["k_factor"] = 10 * (matches["tournament_weight"] + 1)

ratings = defaultdict(lambda: START_ELO)
pre_home_elo, pre_away_elo = [], []
loop_cols = ["home_team", "away_team", "home_score", "away_score", "k_factor", "neutral"]

for row in matches[loop_cols].itertuples(index=False):
    home_elo = ratings[row.home_team]
    away_elo = ratings[row.away_team]
    pre_home_elo.append(home_elo)
    pre_away_elo.append(away_elo)

    delta = elo_delta(
        home_elo, away_elo, row.home_score, row.away_score, row.k_factor, row.neutral
    )
    ratings[row.home_team] = home_elo + delta
    ratings[row.away_team] = away_elo - delta

matches["home_elo_pre"] = pre_home_elo
matches["away_elo_pre"] = pre_away_elo

print(f"Matches processed: {len(matches)}")
print(f"Teams rated: {len(ratings)}")

Matches processed: 48272
Teams rated: 298


Flag competitive matches (anything that is not a Friendly), drop friendlies, then keep only pre-match features and the final score. No post-game Elo column is created or saved.

In [ ]:
matches["competitive"] = matches["tournament"] != "Friendly"
print(matches["competitive"].value_counts())

training = (
    matches.loc[matches["competitive"], [
        "date",
        "home_team",
        "away_team",
        "home_elo_pre",
        "away_elo_pre",
        "city",
        "country",
        "neutral",
        "tournament_weight",
        "home_score",
        "away_score",
    ]]
    .reset_index(drop=True)
)

assert "home_elo_post" not in training.columns
assert "away_elo_post" not in training.columns
assert (matches.loc[~matches["competitive"], "tournament"] == "Friendly").all()

training.to_csv("../data/training/training_matches.csv", index=False)

print(training.shape)
print(list(training.columns))
training.head()

(48272, 11)
['date', 'home_team', 'away_team', 'home_elo_pre', 'away_elo_pre', 'city', 'country', 'neutral', 'tournament_weight', 'home_score', 'away_score']


,date,home_team,away_team,home_elo_pre,away_elo_pre,city,country,neutral,tournament_weight,home_score,away_score
0,1872-11-30,Scotland,England,1500.000000,1500.000000,Glasgow,Scotland,False,1,0.0,0.0
1,1873-03-08,England,Scotland,1502.801300,1497.198700,London,England,False,1,4.0,2.0
2,1874-03-07,Scotland,England,1486.622531,1513.377469,Glasgow,Scotland,False,1,2.0,1.0
3,1875-03-06,England,Scotland,1505.454946,1494.545054,London,England,False,1,2.0,2.0
4,1876-03-04,Scotland,England,1497.633108,1502.366892,Glasgow,Scotland,False,1,3.0,0.0


# Merge confederation data

Add confederation data to training_matches table. This helps determine team strength because teams in certain confederations (UEFA for a european team) are stronger than teams in others. When a team in a weaker confederation and a high elo faces a team in a stronger confederation and high elo, the model will be able to accurately scale team strength, based on confederation.  


If a team is not in raw/fifa_confederations.csv their confederation will be listed as 'Unknown'

In [ ]:
confederations = pd.read_csv("../data/raw/fifa_confederations.csv")
confed_by_nation = confederations.set_index("nation")["confederation"]

training["home_confederation"] = training["home_team"].map(confed_by_nation).fillna("Unknown")
training["away_confederation"] = training["away_team"].map(confed_by_nation).fillna("Unknown")

training = training[[
    "date",
    "home_team",
    "away_team",
    "home_confederation",
    "away_confederation",
    "home_elo_pre",
    "away_elo_pre",
    "city",
    "country",
    "neutral",
    "tournament_weight",
    "home_score",
    "away_score",
]]

training.to_csv("../data/training/training_matches.csv", index=False)

print(training.shape)
print(list(training.columns))
print("\nHome confederations:")
print(training["home_confederation"].value_counts())
print("\nAway confederations:")
print(training["away_confederation"].value_counts())
training.head()